# Etapa 9 — Evaluación en el conjunto de prueba

**Este cuaderno se corre una sola vez**, sobre el commit con la etiqueta `congelado-test` (plan de evaluación, secciones 8, 12 y 13), y se commitea con sus salidas tal como quedan. Es el único lugar donde se abre el conjunto de prueba (`load_test_case`; lo controla `tests/test_lock.py`).

**Qué se evalúa.** El algoritmo congelado (`segment_final`): verde a la mitad de resolución → corrección del fondo → 26 mapas de la etapa 4 → regresión logística (pesos ajustados con las 30 imágenes de desarrollo, `models/pesos_combinacion.json`) → probabilidad → **valor fijo** (etapa 8, desvío 14.7). No se ajusta nada con prueba.

**Comparaciones** (plan, sección 11.1):
- **Algoritmo final.**
- **Referencia clásica:** el detector de líneas solo sobre el verde corregido, con la misma regla de umbral (su propio valor fijo, elegido con desarrollo; desvío 14.9).
- **Verde con Otsu:** la estrategia de referencia del anteproyecto, leída como el canal verde invertido (vasos claros) con el umbral de Otsu dentro del campo de visión detectado, sin filtros.

**Reglas (fijadas en el plan antes de abrir prueba).**
- Medidas por imagen (sección 4), a resolución original y dentro del campo de visión de la base; resumen con la media de las 15 imágenes.
- Intervalos al 95 % con **10.000 remuestreos de personas** (10 personas), semilla 2026. Valor p: el doble de la menor fracción de remuestreos con diferencia ≤ 0 o ≥ 0 (sección 6).
- **Pregunta principal (sección 10):** Matthews del algoritmo final menos el de la referencia. Se confirma si p < 0,05 a favor del algoritmo; se rechaza si p < 0,05 a favor de la referencia; si no, no se muestra diferencia. Con 10 personas la potencia es baja: un resultado sin diferencia no quiere decir que sean iguales.
- **Resultados secundarios (sección 11):** todas las medidas para los tres métodos; por grupo (con la diferencia retinopatía − sanos); por nivel de compresión; medidas clínicas (concordancia con los expertos y diferencia entre sanos y retinopatía); curvas de robustez de la etapa 8 repetidas sobre prueba; literatura como referencia.
- **Después de abrir prueba no se cambia nada** para mejorar un resultado. Si aparece un error de código, se corrige, se declara en la sección 14 del plan y se informan las dos corridas (sección 12).

**Si la corrida se corta** (por ejemplo, la computadora se suspende), las pasadas largas guardan un CSV después de cada imagen y al volver a correr la celda siguen desde donde quedaron. Eso no es abrir prueba de nuevo: se completa la misma corrida, con el mismo código.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True).stdout.strip()


# Control de la versión: tiene que ser el commit `congelado-test` sin cambios en el código.
TAG = "congelado-test"
commit = git("rev-parse", "HEAD")
tags = git("tag", "--points-at", "HEAD").split()
dirty = [line for line in git("status", "--porcelain", "--untracked-files=no").splitlines()
         if not line.endswith("99_prueba.ipynb")]
print("Proyecto:", ROOT)
print("Commit:", commit)
print("Etiquetas del commit:", tags or "ninguna")
assert TAG in tags, f"Este commit no tiene la etiqueta {TAG}: hacer `git checkout {TAG}` antes de correr."
assert not dirty, f"Hay cambios sin commitear en archivos versionados: {dirty}"

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config as C
from src.clinical import clinical_measures
from src.data import heldout_ids, load_split, load_test_case
from src.degrade import gaussian_blur, gaussian_noise, jpeg_recompress, restore_photo
from src.enhance import upscale
from src.fov import detect_fov_robust
from src.metrics import (accuracy, auc_by_subset, centerline_dice, centerline_precision,
                         centerline_sensitivity, centerline_sensitivity_by_class, dice, matthews,
                         sensitivity, specificity, thin_masks)
from src.pipeline import feature_scores, load_threshold, segment_final
from src.segment import segmentar_otsu
from src.stats import (bland_altman, icc_absolute, percentile_interval, person_bootstrap_means,
                       person_bootstrap_stat)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

REPS = 10_000                      # remuestreos en prueba (plan, sección 6)
ids = heldout_ids()
split = load_split().set_index("image").loc[ids]
persons = split["person"]
assert len(ids) == 15 and persons.nunique() == 10

threshold = load_threshold()
assert C.SEGMENT_THRESHOLD_RULE == "valor fijo"
assert threshold["value"] is not None and threshold["reference_value"] is not None
print(f"Regla de umbral: {C.SEGMENT_THRESHOLD_RULE}; algoritmo final > {threshold['value']:.4f}, "
      f"referencia > {threshold['reference_value']:.4f}")
print("Imágenes de prueba:", " ".join(ids))
print(split.groupby("group").agg(imagenes=("person", "size"), personas=("person", "nunique"),
                                 mas_comprimidas=("more_compressed", "sum")))

OUT = ROOT / "results" / "99_prueba"
CACHE = OUT / "mascaras"
CACHE.mkdir(parents=True, exist_ok=True)
METHODS = ["Algoritmo final", "Referencia clásica", "Verde con Otsu"]

## Pasada 1: segmentar las 15 imágenes y medir

Para cada imagen: las máscaras de los tres métodos y sus medidas (sección 4 y desvío 14.3), el área bajo la curva sobre el mapa continuo de cada uno (en todos los vasos y en los finos) y las medidas clínicas de los expertos, del algoritmo final y de la referencia. Las máscaras se guardan en `results/99_prueba/mascaras/` para las figuras.

In [ ]:
REF = C.SEGMENT_REFERENCE_FILTER


def mask_measures(pred, truth, fov):
    by_class = centerline_sensitivity_by_class(pred, truth, C.WIDTH_LIMITS_PX, fov)
    return {"Matthews": matthews(pred, truth, fov), "Dice": dice(pred, truth, fov),
            "Sensibilidad": sensitivity(pred, truth, fov), "Especificidad": specificity(pred, truth, fov),
            "Exactitud": accuracy(pred, truth, fov),
            "Dice línea central": centerline_dice(pred, truth, fov),
            "Sensibilidad línea central": centerline_sensitivity(pred, truth, fov),
            "Precisión línea central": centerline_precision(pred, truth, fov),
            "Sensibilidad finos": by_class["fino"], "Sensibilidad medianos": by_class["mediano"],
            "Sensibilidad gruesos": by_class["grueso"]}


def score_measures(score, truth, fov, thin):
    a = auc_by_subset(score, truth, {"finos": thin}, region=fov)
    return {"AUC": a["todos"], "AUC finos": a["finos"]}


def reference_score(rgb, shape):
    # El mismo mapa que umbraliza segment_final(reference=True): el detector de líneas estandarizado.
    _, kept, _ = feature_scores(rgb, [], keep=(REF,))
    return upscale(kept[REF], shape)


def simple_green(rgb):
    # Estrategia del anteproyecto (plan, 11.1): verde invertido (vasos claros) y Otsu dentro del campo detectado.
    score = 255.0 - rgb[..., 1].astype(np.float32)
    return segmentar_otsu(score, detect_fov_robust(rgb))[0], score


def first_pass(image_id):
    rgb, truth, fov = load_test_case(image_id)
    mask_final, prob = segment_final(rgb, return_score=True)
    mask_ref = segment_final(rgb, reference=True)
    score_ref = reference_score(rgb, truth.shape)
    mask_green, score_green = simple_green(rgb)
    thin = thin_masks(truth, fov)[0]
    rows = [{"Imagen": image_id, "Método": name, **mask_measures(m, truth, fov),
             **score_measures(s, truth, fov, thin)}
            for name, m, s in (("Algoritmo final", mask_final, prob),
                               ("Referencia clásica", mask_ref, score_ref),
                               ("Verde con Otsu", mask_green, score_green))]
    clinical = [{"Imagen": image_id, "Fuente": name, **clinical_measures(m, fov)}
                for name, m in (("Experto", truth), ("Algoritmo final", mask_final),
                                ("Referencia clásica", mask_ref))]
    np.savez_compressed(CACHE / f"{image_id}.npz", final=np.packbits(mask_final),
                        reference=np.packbits(mask_ref), shape=np.array(mask_final.shape))
    return rows, clinical


MEASURES_CSV = OUT / "99_medidas.csv"
CLINICAL_CSV = OUT / "99_clinicas.csv"
measure_rows = pd.read_csv(MEASURES_CSV).to_dict("records") if MEASURES_CSV.exists() else []
clinical_rows = pd.read_csv(CLINICAL_CSV).to_dict("records") if CLINICAL_CSV.exists() else []
done = {r["Imagen"] for r in measure_rows}
for n, image_id in enumerate(ids, start=1):
    if image_id in done:
        continue
    t0 = time.time()
    rows, clinical = first_pass(image_id)
    measure_rows += rows
    clinical_rows += clinical
    pd.DataFrame(measure_rows).to_csv(MEASURES_CSV, index=False)
    pd.DataFrame(clinical_rows).to_csv(CLINICAL_CSV, index=False)
    print(f"{n}/{len(ids)} — {image_id}: {time.time() - t0:.0f} s", flush=True)

R = pd.DataFrame(measure_rows).join(split[["group", "person", "more_compressed"]], on="Imagen")
K = pd.DataFrame(clinical_rows)
assert set(R["Imagen"]) == set(ids)
print(R.groupby("Método")[["Matthews", "Dice", "Dice línea central", "AUC"]].mean().loc[METHODS].round(4))

## Pregunta principal (plan, sección 10)

¿El algoritmo final segmenta mejor que el método clásico de un solo mapa? Matthews por imagen, algoritmo final menos referencia, sobre las mismas 15 imágenes.

In [ ]:
def by_image(measure, methods=METHODS):
    return R.pivot(index="Imagen", columns="Método", values=measure).loc[ids, methods]


def interval(values):
    # Media e intervalo al 95 % de una columna por imagen, remuestreando personas.
    boot = person_bootstrap_means(values.to_frame(), persons.loc[values.index], reps=REPS)[:, 0]
    lo, hi = percentile_interval(boot)
    return float(values.mean()), float(lo), float(hi), boot


mcc = by_image("Matthews")
diff = mcc["Algoritmo final"] - mcc["Referencia clásica"]
mean, lo, hi, boot = interval(diff)
p_value = min(1.0, 2 * min((boot <= 0).mean(), (boot >= 0).mean()))
if p_value < 0.05:
    verdict = ("SE CONFIRMA: el algoritmo final segmenta mejor que la referencia." if mean > 0
               else "SE RECHAZA: la referencia segmenta mejor que el algoritmo final.")
else:
    verdict = "NO SE MUESTRA DIFERENCIA con estos datos (con 10 personas la potencia es baja)."

print(f"Matthews medio: algoritmo final {mcc['Algoritmo final'].mean():.4f}, "
      f"referencia {mcc['Referencia clásica'].mean():.4f}")
print(f"Diferencia: {mean:+.4f} (intervalo al 95 %: {lo:+.4f} a {hi:+.4f}); p = {p_value:.4f}")
print(f"El algoritmo final gana en {(diff > 0).sum()} de 15 imágenes y en "
      f"{(diff.groupby(persons).mean() > 0).sum()} de 10 personas.")
print(verdict)
print("\nEn desarrollo (validación cruzada por personas, 05d, valor fijo): algoritmo final 0,7302, "
      "referencia 0,7040 (+0,026).")

## Resultados secundarios

### 1. Todas las medidas, por método (plan, 11.1)

Media de las 15 imágenes e intervalo al 95 % por personas. Abajo, la diferencia apareada algoritmo final − referencia en cada medida.

In [ ]:
MEASURES = ["Matthews", "Dice", "Sensibilidad", "Especificidad", "Exactitud", "AUC", "AUC finos",
            "Dice línea central", "Sensibilidad línea central", "Precisión línea central",
            "Sensibilidad finos", "Sensibilidad medianos", "Sensibilidad gruesos"]


def summary(frame):
    boot = person_bootstrap_means(frame, persons.loc[frame.index], reps=REPS)
    lo, hi = percentile_interval(boot)
    return pd.DataFrame({"media": frame.mean(), "inferior": lo, "superior": hi}, index=frame.columns)


per_method = {m: summary(R[R["Método"] == m].set_index("Imagen").loc[ids, MEASURES]) for m in METHODS}
all_measures = pd.concat(per_method, axis=1)
all_measures.to_csv(OUT / "99_todas_las_medidas.csv")
display(all_measures.round(4))

paired = (R[R["Método"] == "Algoritmo final"].set_index("Imagen").loc[ids, MEASURES]
          - R[R["Método"] == "Referencia clásica"].set_index("Imagen").loc[ids, MEASURES])
print("Algoritmo final − referencia clásica:")
display(summary(paired).round(4))

### 2. Por grupo (plan, 11.2)

Media por grupo y diferencia retinopatía − sanos, remuestreando personas dentro de cada grupo (diferencia no apareada, sección 6).

In [ ]:
def group_difference(values, a="dr", b="h"):
    va = values[split["group"] == a]
    vb = values[split["group"] == b]
    ba = person_bootstrap_means(va.to_frame(), persons.loc[va.index], reps=REPS, seed=C.SEED)[:, 0]
    bb = person_bootstrap_means(vb.to_frame(), persons.loc[vb.index], reps=REPS, seed=C.SEED + 1)[:, 0]
    lo, hi = percentile_interval(ba - bb)
    return {"retinopatía − sanos": va.mean() - vb.mean(), "inferior": float(lo), "superior": float(hi)}


GROUP_NAMES = {"h": "Sanos", "dr": "Retinopatía diabética", "g": "Glaucoma"}
for measure in ("Matthews", "Dice línea central", "Sensibilidad finos"):
    t = by_image(measure)
    means = t.groupby(split["group"]).mean().rename(index=GROUP_NAMES)
    print(f"\n{measure}: media por grupo")
    display(means.round(4))
    display(pd.DataFrame({m: group_difference(t[m]) for m in METHODS}).T.round(4))

### 3. Por nivel de compresión (plan, 11.3)

Descriptivo. Compresión y grupo están mezclados, y hay pocas imágenes por celda.

In [ ]:
LEVEL = split["more_compressed"].map({True: "Más comprimidas", False: "Menos comprimidas"})
print(pd.crosstab(split["group"].map(GROUP_NAMES), LEVEL))
for measure in ("Matthews", "Dice línea central"):
    print(f"\n{measure}:")
    display(by_image(measure).groupby(LEVEL).mean().round(4))

### 4. Medidas clínicas (plan, 11.4)

Con los parámetros de `07a`. Concordancia de cada máscara con la de los expertos (correlación intraclase de acuerdo absoluto y Bland–Altman, algoritmo − experto) y diferencia entre sanos y retinopatía calculada con cada máscara.

In [ ]:
CLIN = ["proporción", "longitud", "tortuosidad"]
SOURCES = ["Algoritmo final", "Referencia clásica"]
frame = pd.concat({m: K.pivot(index="Imagen", columns="Fuente", values=m) for m in CLIN}, axis=1).loc[ids]
print("Media por fuente:")
display(K.groupby("Fuente")[CLIN + ["tramos"]].mean().loc[["Experto"] + SOURCES].round(4))


def iccs(f):
    return np.array([icc_absolute(f[(m, "Experto")], f[(m, s)]) for s in SOURCES for m in CLIN])


boot_icc = person_bootstrap_stat(frame, persons, iccs, reps=REPS)
lo, hi = percentile_interval(boot_icc)
point = iccs(frame)
rows = []
for j, (s, m) in enumerate([(s, m) for s in SOURCES for m in CLIN]):
    rows.append({"fuente": s, "medida": m, "correlación intraclase": point[j], "inferior": lo[j],
                 "superior": hi[j], **bland_altman(frame[(m, "Experto")], frame[(m, s)])})
agreement = pd.DataFrame(rows).set_index(["fuente", "medida"])
agreement.to_csv(OUT / "99_concordancia_clinica.csv")
display(agreement.round(4))

mean_icc = boot_icc.reshape(len(boot_icc), len(SOURCES), len(CLIN)).mean(axis=2)
lo_m, hi_m = percentile_interval(mean_icc)
print("Correlación intraclase media en las tres medidas:")
print(pd.DataFrame({"media": point.reshape(len(SOURCES), len(CLIN)).mean(axis=1), "inferior": lo_m,
                    "superior": hi_m}, index=SOURCES).round(4).to_string())
print("En desarrollo (07b, valor fijo): algoritmo final 0,21.")

print("\nSanos − retinopatía con cada máscara:")
rows = []
for m in CLIN:
    for s in ["Experto"] + SOURCES:
        d = group_difference(frame[(m, s)], a="h", b="dr")
        rows.append({"medida": m, "fuente": s, "sanos − retinopatía": d["retinopatía − sanos"],
                     "inferior": d["inferior"], "superior": d["superior"]})
display(pd.DataFrame(rows).set_index(["medida", "fuente"]).round(4))

print("Por nivel de compresión y grupo (descriptivo):")
display(K.join(split[["group", "more_compressed"]], on="Imagen")
        .pivot_table(index=["more_compressed", "group"], columns="Fuente", values=CLIN).round(4))

### 5. Robustez sobre prueba (plan, 11.5)

Las degradaciones y restauraciones de la etapa 8, con los mismos niveles y la misma semilla del ruido, aplicadas al algoritmo congelado. La foto original sin restauración es la de la pasada 1. Un nivel se tolera si el límite inferior del intervalo de (nivel − original) es ≥ −0,03; la restauración ayuda si el límite inferior de (con − sin) es > 0.

**Tarda:** unas 20 segmentaciones por imagen (2 a 3 horas en total). Se reanuda si se corta.

In [ ]:
SIGMAS_PX = [round(r * C.FOV_DIAMETER_REF, 3) for r in C.ROBUST_BLUR_SIGMA_REL]
FAMILIES = {
    "JPEG": [(f"JPEG {q}", lambda f, i, q=q: jpeg_recompress(f, q), "bloques", {})
             for q in C.ROBUST_JPEG_QUALITIES],
    "Desenfoque": [(f"σ = {s:g} px", lambda f, i, s=s: gaussian_blur(f, s), "wiener_desenfoque",
                    {"blur_sigma_px": s}) for s in SIGMAS_PX],
    "Ruido": [(f"σ = {s} niveles", lambda f, i, s=s: gaussian_noise(f, s, seed=C.SEED + sum(map(ord, i)) + s),
               "wiener_ruido", {}) for s in C.ROBUST_NOISE_SIGMAS],
}
ROBUST_CSV = OUT / "99_robustez.csv"
robust_rows = pd.read_csv(ROBUST_CSV).to_dict("records") if ROBUST_CSV.exists() else []
done = {(r["Imagen"], r["Familia"], r["Nivel"], r["Restauración"]) for r in robust_rows}

for n, image_id in enumerate(ids, start=1):
    t0 = time.time()
    rgb, truth, fov = load_test_case(image_id)
    jobs = [("JPEG", "Original", lambda f, i: f, "bloques", {}, ("con",))]   # suavizado de bloques sobre la original
    jobs += [(fam, name, degrade, kind, params, ("sin", "con"))
             for fam, levels in FAMILIES.items() for name, degrade, kind, params in levels]
    for fam, name, degrade, kind, params, modes in jobs:
        pending = [r for r in modes if (image_id, fam, name, r) not in done]
        if not pending:
            continue
        degraded = degrade(rgb, image_id)
        for r in pending:
            photo = degraded if r == "sin" else restore_photo(degraded, kind, **params)
            mask = segment_final(photo)
            robust_rows.append({"Imagen": image_id, "Familia": fam, "Nivel": name, "Restauración": r,
                                "Matthews": matthews(mask, truth, fov), "Dice": dice(mask, truth, fov)})
            done.add((image_id, fam, name, r))
        pd.DataFrame(robust_rows).to_csv(ROBUST_CSV, index=False)
    print(f"{n}/{len(ids)} — {image_id}: {time.time() - t0:.0f} s", flush=True)

In [ ]:
T = pd.DataFrame(robust_rows)
base = R[R["Método"] == "Algoritmo final"].set_index("Imagen").loc[ids, "Matthews"]
ORDER = {"JPEG": ["Original"] + [f"JPEG {q}" for q in C.ROBUST_JPEG_QUALITIES],
         "Desenfoque": ["Original"] + [f"σ = {s:g} px" for s in SIGMAS_PX],
         "Ruido": ["Original"] + [f"σ = {s} niveles" for s in C.ROBUST_NOISE_SIGMAS]}


def table(family, mode):
    t = T[(T["Familia"] == family) & (T["Restauración"] == mode)]
    w = t.pivot(index="Imagen", columns="Nivel", values="Matthews").reindex(ids)
    if mode == "sin":
        w["Original"] = base
    return w


rows_a, rows_b = [], []
for family, levels in ORDER.items():
    off, on = table(family, "sin"), table(family, "con")
    for level in levels[1:]:
        mean, lo, hi, _ = interval(off[level] - off["Original"])
        rows_a.append({"Familia": family, "Nivel": level, "Matthews": off[level].mean(),
                       "Diferencia con la original": mean, "inferior": lo, "superior": hi,
                       "Tolerado": lo >= -C.NONINF_MARGIN_MATTHEWS})
    for level in levels:
        if level not in on:
            continue
        mean, lo, hi, _ = interval(on[level] - off[level])
        effect = "ayuda" if lo > 0 else "empeora" if hi < 0 else "sin diferencia demostrada"
        rows_b.append({"Familia": family, "Nivel": level, "Sin": off[level].mean(), "Con": on[level].mean(),
                       "Con − sin": mean, "inferior": lo, "superior": hi, "Efecto": effect})
range_test = pd.DataFrame(rows_a)
restoration_test = pd.DataFrame(rows_b)
range_test.to_csv(OUT / "99_robustez_rango.csv", index=False)
restoration_test.to_csv(OUT / "99_robustez_restauracion.csv", index=False)
print("Rango tolerado (sin restauración):")
display(range_test.round(4))
print("Efecto de la restauración:")
display(restoration_test.round(4))
print("En desarrollo (etapa 8, valor fijo): tolera JPEG hasta calidad 50 y desenfoque hasta σ = 4 px; "
      "con ruido, hasta σ = 5 niveles.")

## Figuras para el informe

1. Curvas de robustez sobre prueba.
2. Ejemplo cualitativo: la imagen de sanos y la de retinopatía con el Matthews **mediano** de su grupo con el algoritmo final (regla fija, para no elegir el ejemplo a ojo). Verde: acierto; rojo: vaso omitido; azul: falsa detección, dentro del campo de visión de la base.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (family, levels) in zip(axes, ORDER.items()):
    for mode, style in (("sin", "o-"), ("con", "s--")):
        w = table(family, mode)
        if mode == "con" and "Original" not in w:
            w["Original"] = base
        ax.plot(range(len(levels)), w.reindex(columns=levels).mean(), style, label=f"{mode} restauración")
    ax.set_xticks(range(len(levels)), levels, rotation=20)
    ax.set_title(family)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Matthews medio (prueba)")
axes[0].legend()
plt.tight_layout()
plt.savefig(OUT / "99_robustez_curvas.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
final = R[R["Método"] == "Algoritmo final"].set_index("Imagen")
examples = []
for g in ("h", "dr"):
    s = final.loc[split.index[split["group"] == g], "Matthews"].sort_values()
    examples.append(s.index[len(s) // 2])

fig, axes = plt.subplots(len(examples), 3, figsize=(15, 4.2 * len(examples)))
for row, image_id in zip(axes, examples):
    rgb, truth, fov = load_test_case(image_id)
    saved = np.load(CACHE / f"{image_id}.npz")
    shape = tuple(saved["shape"])
    mask = np.unpackbits(saved["final"])[: shape[0] * shape[1]].reshape(shape).astype(bool)
    errors = np.zeros((*shape, 3), dtype=np.uint8)
    errors[mask & truth & fov] = [0, 200, 0]
    errors[~mask & truth & fov] = [230, 0, 0]
    errors[mask & ~truth & fov] = [0, 130, 255]
    row[0].imshow(rgb)
    row[0].set_title(f"{image_id} ({GROUP_NAMES[split.loc[image_id, 'group']]})")
    row[1].imshow(truth, cmap="gray")
    row[1].set_title("Expertos")
    row[2].imshow(errors)
    row[2].set_title(f"Algoritmo final (Matthews {final.loc[image_id, 'Matthews']:.3f})")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "99_ejemplos.png", dpi=150, bbox_inches="tight")
plt.show()

## Literatura (plan, 11.6)

Como referencia y no como meta. Valores que informan los autores en esta base (completar desde los artículos, citando tabla y página):

| Trabajo | Sensibilidad | Especificidad | Exactitud |
|---|---|---|---|
| Budai et al. (2013) | | | |
| Odstrcilik et al. (2013) | | | |
| Este trabajo, prueba (algoritmo final) | ver la tabla de la sección 1 | | |

## Resultados (escrito después de correr)

*Completar con los números de arriba, sin cambiar nada del algoritmo ni de las medidas.*

- **Pregunta principal:** …
- **Comparación con desarrollo:** …
- **Por grupo:** …
- **Medidas clínicas:** …
- **Robustez:** …
- **Limitaciones:** 15 imágenes y 10 personas (potencia baja); las mismas limitaciones de desarrollo (vasos finos, retinopatía; desvío 14.10).